# NOORIX-AI Everyday Object & Person Detection — YOLOv8n
**Prepared by Muhammad Farhan**

This notebook uses the COCO-pretrained YOLOv8 nano model for everyday objects and people. COCO already includes the `person` class, so a separate person model is not normally needed for basic person detection. Optional custom training is supported if you have a labelled YOLO dataset.

## Kaggle notebook title
`NOORIX-Object-Person-YOLOv8n-Muhammad-Farhan`

## Dataset to add
- For pretrained COCO inference/export: **no dataset input is required**; model weights are fetched by Ultralytics when internet access is enabled.
- For custom training: add your own YOLO-format dataset with `data.yaml`, `images/train`, `images/val`, `labels/train`, `labels/val`. Kaggle search term: `COCO 2017 object detection YOLO format`. Ultralytics COCO documentation describes the 80 everyday classes, including person.

Do not assume this YOLO ONNX output plugs directly into NOORIX's current SSD adapter. A YOLO-specific decoder/adapter and numerical validation are required before app integration.


In [ ]:
!pip -q install ultralytics onnx
from pathlib import Path
import json, shutil, torch
from ultralytics import YOLO
print("GPU:",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# First run pretrained COCO model. YOLOv8n is the lightweight option for CPU deployment experiments.
model=YOLO("yolov8n.pt")
print("COCO classes:",len(model.names))
print("Person class id:",[k for k,v in model.names.items() if v=="person"])


In [ ]:
# Optional custom training. Leave CUSTOM_DATA_YAML empty to skip training and use COCO pretrained weights.
CUSTOM_DATA_YAML="" # e.g. "/kaggle/input/my-yolo-data/data.yaml"
if CUSTOM_DATA_YAML:
    model.train(data=CUSTOM_DATA_YAML,epochs=30,imgsz=640,batch=16,device=0 if torch.cuda.is_available() else "cpu",
                project="/kaggle/working/noorix_yolo",name="custom",patience=8)
    model=YOLO("/kaggle/working/noorix_yolo/custom/weights/best.pt")
else:
    print("Skipping custom training; using pretrained COCO model.")


In [ ]:
# Evaluate on a sample image or dataset. Set TEST_IMAGE to a file from an attached image dataset.
TEST_IMAGE=""
if TEST_IMAGE:
    results=model.predict(TEST_IMAGE,conf=0.35,save=True)
    for r in results:
        for b in r.boxes:
            print(model.names[int(b.cls[0])],float(b.conf[0]),b.xyxy[0].tolist())
else:
    print("Set TEST_IMAGE to inspect a sample, or attach a labelled dataset and use model.val(data=...).")


In [ ]:
# Export ONNX and package the .pt + ONNX + class metadata.
exported=model.export(format="onnx",imgsz=640,dynamic=False)
out=Path("/kaggle/working/NOORIX_Object_Person_YOLOv8n_Muhammad_Farhan"); out.mkdir(exist_ok=True)
shutil.copy2(model.ckpt_path,out/"best.pt") if getattr(model,"ckpt_path",None) and Path(model.ckpt_path).exists() else shutil.copy2("yolov8n.pt",out/"yolov8n.pt")
shutil.copy2(exported,out/Path(exported).name)
(out/"labels.json").write_text(json.dumps({str(k):v for k,v in model.names.items()},indent=2),encoding="utf-8")
(out/"metadata.json").write_text(json.dumps({"project":"NOORIX-AI","prepared_by":"Muhammad Farhan","model":"YOLOv8n","classes":len(model.names),
"input_size":[1,3,640,640],"note":"YOLO ONNX needs its own decoder; not compatible with the current SSD output adapter."},indent=2),encoding="utf-8")
z=shutil.make_archive("/kaggle/working/NOORIX_Object_Person_YOLOv8n_Muhammad_Farhan","zip",out)
print("ZIP:",z)
